# Fine-tuning Llama-3-8B using LoRA

### The goal of this project is to fine-tune a model to perform better on classifying text as 'toxic' or 'not toxic'. The process will involve first evaluating a baseline score for how the model performs on identifying toxic prompts. Then, I will use LoRA to fine-tune the model and then evaluate its performance on identify toxic prompts.

In [ ]:
! git clone https://github.com/pxd222PranavDhinakar/LoRA_Stacking.git
! pip install datasets
! pip install peft
! pip install trl

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
%cd LoRA_Stacking/

In [ ]:
# Install the Hugging Face CLI
! pip install --quiet huggingface_hub

# Log in to Hugging Face (you'll be prompted to enter your token)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
! apt-get install git-lfs
! GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/meta-llama/Meta-Llama-3-8B

I'm having a problem when using GIT_LFS_SKIP_SMUDGE=1. If this is a problem you will have to download the files fully so use this to make all the files download fully. Could also add these files to your google drive to make things go quicker for future runs as these session files will not persist.

In [ ]:
%cd Meta-Llama-3-8B
! git lfs pull

In [ ]:
%cd ..

In [ ]:
dataset = load_dataset("google/jigsaw_toxicity_pred", data_dir="/content/drive/MyDrive/jigsaw-toxic-comment-classification-challenge")

In [ ]:
dataset["train"]

In [ ]:
# Model and tokenizer paths
model_path = "./Meta-Llama-3-8B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# need to pad the input with EOS token
def preprocess_test(examples):
    return tokenizer(examples["comment_text"], truncation=True, padding="max_length", max_length=256)

tokenized_dataset = dataset["test"].map(preprocess_test, batched=True)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
  model_path,
  device_map="auto",
  torch_dtype=torch.float16,
  trust_remote_code=True,
)

model.eval()

Simple test - just testing whether the text is Toxic or not. Not looking at other labels. Just to get a simple baseline.

In [ ]:
def run_inference(model, tokenizer, dataset):
  # sample text from the dataset
  text = dataset["test"][2]['comment_text'] # Replace with different samples

  # prompt for classification
  prompt = f"""You are a toxicity classifier. Read the following text and strictly classify it as either 'toxic' or 'not toxic'.
  Only output 'toxic' or 'not toxic' with no explanation.

  Example 1:
  Text: "You are a cocksucker."
  Answer: toxic

  Example 2:
  Text: "You are a nice person."
  Answer: not toxic

  Now classify the following text:
  Text: "{text}"
  Answer: """


  # tokenize and move input to the same device as the model
  inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to("cuda")

  # generate response
  with torch.no_grad():
      output = model.generate(**inputs, max_new_tokens=5)  # Limit response length

  full_response = tokenizer.decode(output[0], skip_special_tokens=True)
  generated_text = full_response[len(prompt):].strip()

  print(generated_text)

run_inference(model, tokenizer, dataset)

Evaluating the model before fine-tuning to evaluate baseline performance

In [ ]:
def classify_toxicity(text):
    # generate a classification of the text
    # TODO: find a better prompt. Getting mixed results with this because the model isn't always outputting the answer in the format specified.
    # might have to try doing n-shot prompting.
    prompt = f"""You are a toxicity classifier. Read the following text and strictly classify it as either 'toxic' or 'not toxic'.
    Only output 'toxic' or 'not toxic' with no explanation.

    Example 1:
    Text: "You are a cocksucker."
    Answer: toxic

    Example 2:
    Text: "You are a nice person."
    Answer: not toxic

    Now classify the following text:
    Text: "{text}"
    Answer: """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to("cuda")

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5)

    full_response = tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()
    response = full_response[len(prompt):].strip()

    if "not toxic" in response:
        return "not toxic"
    elif "toxic" in response:
        return "toxic"
    else:
        return "unknown"

# convert dataset labels into binary classes (toxic vs. not toxic)
def get_ground_truth_labels(dataset):
    # maps toxicity labels to 'toxic' (1) or 'not toxic' (0).
    return ["toxic" if example['toxic'] >= 0.5 else "not toxic" for example in dataset]


subset_size = 300
test_data_subset = dataset["test"].select(range(subset_size))

# evaluate the model
y_true = get_ground_truth_labels(test_data_subset)
y_pred = []

for example in tqdm(test_data_subset, desc="Evaluating"):
    prediction = classify_toxicity(example["comment_text"])
    y_pred.append(prediction)

# convert the numerical labels for evaluation
y_true_bin = [1 if label == "toxic" else 0 for label in y_true]
y_pred_bin = [1 if label == "toxic" else 0 for label in y_pred]

# calculate scores
accuracy = accuracy_score(y_true_bin, y_pred_bin)
precision, recall, f1, _ = precision_recall_fscore_support(y_true_bin, y_pred_bin, average="binary")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

We now have a baseline to compare a fine-tuned model against. We can now continue and configure LoRA and then train and evaluate the model using the same evaluation as before it was fine-tuned.

In [ ]:
# small subset for training and evaluation
train_subset = dataset["train"].select(range(900))
test_subset = dataset["test"].select(range(300))

def preprocess(examples):
  # Tokenize the text
  inputs = tokenizer(
      examples["comment_text"],
      truncation=True,
      padding="max_length",
      max_length=128
  )

  # Use only the "toxic" column as label
  inputs["labels"] = examples["toxic"]

  return inputs

def compute_metrics(pred):
    predictions = np.argmax(pred.predictions, axis=-1)
    return {"accuracy": (predictions == pred.label_ids).mean()}

lora_config = LoraConfig(
    r=32,  #  rank
    lora_alpha=64,  #  alpha
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention components
        "gate_proj", "up_proj", "down_proj"  # mlp components
    ],
    lora_dropout=0.05,  # dropout
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Apply preprocessing to training
tokenized_train = train_subset.map(preprocess, batched=True)
tokenized_test = test_subset.map(preprocess, batched=True)

# Define training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    output_dir="./lora_finetuned_toxicity",
    save_total_limit=2,
    fp16=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    args=training_args,
    tokenizer=tokenizer
)

trainer.train()

model.save_pretrained("./lora_finetuned_toxicity")
tokenizer.save_pretrained("./lora_finetuned_toxicity")